# Final Third Zone Dominance with mplsoccer

## Purpose
This notebook analyzes how teams entered the final third by channel and visualizes those entry routes on a football pitch using `mplsoccer`.

## Research Question
Where did teams enter the final third: wide, half-space, or central zones?

## Data Scope
- Source file: `site_official_stats_team_wide_flagged.csv`
- Source: FIFA official Match Centre statistics
- Main filter: `stats_complete == True`
- Excluded match: Belgium vs Egypt (`match_id = 400021478`), because FIFA only provides Live Statistics for that match
- Views: Overall, Group Stage, Knockout Stage

## Key Metrics
- Wide Entry Share: `(left_channel + right_channel) / total final-third entries x 100`
- Half-space Entry Share: `(left_inside_channel + right_inside_channel) / total final-third entries x 100`
- Central Entry Share: `central_channel / total final-third entries x 100`
- Zone Dominance Proxy: team zone entries / both teams' zone entries x 100

## Interpretation Rule
These are entry-based zone metrics. They describe where teams entered the final third, not where they controlled possession continuously. A high zone share is a tactical tendency, not automatically better performance.

## Data Limitations for Publication
- Team match counts differ across the tournament because some teams played 3 matches and finalists played up to 8 matches. Spain-centered rankings are descriptive champion profiling, not a causal model of why Spain won.
- Belgium vs Egypt (`match_id = 400021478`) is excluded from the main analytical tables because FIFA provides only `Live Statistics` for that match. As a result, Belgium has 5 full-stat matches instead of 6, and Egypt has 4 full-stat matches instead of 5. Per-match metrics for those two teams can be slightly inflated because one real match is not in the denominator.




In [ ]:
"""
Step 1: Environment setup and data load
=======================================
Only BASE_DIR should need editing if the project folder moves.
"""

import sys
import subprocess
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from pathlib import Path

try:
    from mplsoccer import Pitch
except ModuleNotFoundError:
    print("[Setup] mplsoccer is not installed in this Jupyter kernel. Installing now...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "mplsoccer"])
    from mplsoccer import Pitch
    print("[Setup] mplsoccer installed successfully.")

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

BASE_DIR = Path.cwd().resolve()
for parent in [BASE_DIR, *BASE_DIR.parents]:
    if parent.name == "worldcup-2026-official-stats-analysis":
        BASE_DIR = parent
        break
DATA_DIR = BASE_DIR / "data" / "fifa_worldcup_2026" / "site_scrape"
PUBLIC_DIR = BASE_DIR
OUTPUT_DATA_DIR = PUBLIC_DIR / "data"
OUTPUT_FIG_DIR = PUBLIC_DIR / "figures"

RAW_CSV_PATH = DATA_DIR / "site_official_stats_team_wide_flagged.csv"
GROUP_STAGE_PATH = DATA_DIR / "site_official_stats_team_wide_group_stage.csv"

for path_name, path_value in {
    "BASE_DIR": BASE_DIR,
    "DATA_DIR": DATA_DIR,
    "PUBLIC_DIR": PUBLIC_DIR,
    "RAW_CSV_PATH": RAW_CSV_PATH,
    "GROUP_STAGE_PATH": GROUP_STAGE_PATH,
}.items():
    if not path_value.exists():
        raise FileNotFoundError(f"{path_name} does not exist: {path_value}")

OUTPUT_DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FIG_DIR.mkdir(parents=True, exist_ok=True)

raw = pd.read_csv(RAW_CSV_PATH)

print("[Raw file]")
print(f"Rows: {len(raw):,}")
print(f"Matches: {raw['match_id'].nunique():,}")
print(f"Teams: {raw['team_name'].nunique():,}")

display(
    raw[["match_id", "stats_source", "stats_complete"]]
    .drop_duplicates()
    .groupby(["stats_source", "stats_complete"])
    .size()
    .reset_index(name="matches")
)

# Keep only full FIFA Official Stats matches. Missing values are not imputed.
df = raw[raw["stats_complete"] == True].copy()

print("\n[Analysis file]")
print(f"Rows: {len(df):,}")
print(f"Matches: {df['match_id'].nunique():,}")
print(f"Teams: {df['team_name'].nunique():,}")





## Build Zone Entry Metrics

The FIFA site provides final-third entries in five channels:

- left channel
- left inside channel
- central channel
- right inside channel
- right channel

For interpretation, they are also grouped into:

- Wide zones
- Half-space zones
- Central zone


In [ ]:
"""
Step 2: Build final-third zone entry metrics by phase
=====================================================
"""

CHAMPION_TEAM = "Spain"

group_stage_match_ids = set(pd.read_csv(GROUP_STAGE_PATH)["match_id"].unique())

df["competition_phase"] = np.where(
    df["match_id"].isin(group_stage_match_ids),
    "Group Stage",
    "Knockout Stage"
)

print("[Phase split]")
display(
    df[["match_id", "competition_phase"]]
    .drop_duplicates()
    .groupby("competition_phase")
    .size()
    .reset_index(name="matches")
)

zone_cols = {
    "left_channel": "attacking__final_third_entries__left_channel",
    "left_inside_channel": "attacking__final_third_entries__left_inside_channel",
    "central_channel": "attacking__final_third_entries__central_channel",
    "right_inside_channel": "attacking__final_third_entries__right_inside_channel",
    "right_channel": "attacking__final_third_entries__right_channel",
}

required_cols = ["match_id", "team_name", "team_side", "opponent_side"] + list(zone_cols.values())
missing_required = [col for col in required_cols if col not in df.columns]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")


def safe_divide(numerator, denominator):
    return np.where(denominator == 0, np.nan, numerator / denominator)


def prepare_zone_rows(input_df):
    temp = input_df.copy()

    for short_name, source_col in zone_cols.items():
        temp[short_name] = temp[source_col]

    temp["wide_entries"] = temp["left_channel"] + temp["right_channel"]
    temp["halfspace_entries"] = temp["left_inside_channel"] + temp["right_inside_channel"]
    temp["central_entries"] = temp["central_channel"]
    temp["final_third_entries_total"] = temp[list(zone_cols.keys())].sum(axis=1, min_count=5)

    return temp


def calculate_zone_metrics(input_df, phase_label):
    temp = prepare_zone_rows(input_df)

    # Opponent values for grouped zone dominance proxy.
    opponent_zone = (
        temp[["match_id", "team_side", "wide_entries", "halfspace_entries", "central_entries", "final_third_entries_total"]]
        .rename(columns={
            "team_side": "opponent_side",
            "wide_entries": "opponent_wide_entries",
            "halfspace_entries": "opponent_halfspace_entries",
            "central_entries": "opponent_central_entries",
            "final_third_entries_total": "opponent_final_third_entries_total",
        })
    )

    temp = temp.merge(
        opponent_zone,
        on=["match_id", "opponent_side"],
        how="left",
        validate="many_to_one",
    )

    valid = temp.dropna(subset=["final_third_entries_total", "opponent_final_third_entries_total"]).copy()

    team = (
        valid
        .groupby("team_name")
        .agg(
            matches=("match_id", "nunique"),
            left_channel=("left_channel", "sum"),
            left_inside_channel=("left_inside_channel", "sum"),
            central_channel=("central_channel", "sum"),
            right_inside_channel=("right_inside_channel", "sum"),
            right_channel=("right_channel", "sum"),
            wide_entries=("wide_entries", "sum"),
            halfspace_entries=("halfspace_entries", "sum"),
            central_entries=("central_entries", "sum"),
            final_third_entries_total=("final_third_entries_total", "sum"),
            opponent_wide_entries=("opponent_wide_entries", "sum"),
            opponent_halfspace_entries=("opponent_halfspace_entries", "sum"),
            opponent_central_entries=("opponent_central_entries", "sum"),
            opponent_final_third_entries_total=("opponent_final_third_entries_total", "sum"),
        )
        .reset_index()
    )

    team["competition_phase"] = phase_label

    for zone in zone_cols.keys():
        team[f"{zone}_share"] = safe_divide(team[zone], team["final_third_entries_total"]) * 100

    team["wide_entry_share"] = safe_divide(team["wide_entries"], team["final_third_entries_total"]) * 100
    team["halfspace_entry_share"] = safe_divide(team["halfspace_entries"], team["final_third_entries_total"]) * 100
    team["central_entry_share"] = safe_divide(team["central_entries"], team["final_third_entries_total"]) * 100

    team["field_tilt_proxy"] = safe_divide(
        team["final_third_entries_total"],
        team["final_third_entries_total"] + team["opponent_final_third_entries_total"]
    ) * 100
    team["wide_zone_dominance_proxy"] = safe_divide(
        team["wide_entries"],
        team["wide_entries"] + team["opponent_wide_entries"]
    ) * 100
    team["halfspace_zone_dominance_proxy"] = safe_divide(
        team["halfspace_entries"],
        team["halfspace_entries"] + team["opponent_halfspace_entries"]
    ) * 100
    team["central_zone_dominance_proxy"] = safe_divide(
        team["central_entries"],
        team["central_entries"] + team["opponent_central_entries"]
    ) * 100

    team["final_third_entries_per_match"] = safe_divide(team["final_third_entries_total"], team["matches"])

    return team


zone_overall = calculate_zone_metrics(df, "Overall")
zone_group_stage = calculate_zone_metrics(df[df["competition_phase"] == "Group Stage"], "Group Stage")
zone_knockout_stage = calculate_zone_metrics(df[df["competition_phase"] == "Knockout Stage"], "Knockout Stage")

zone_metrics_by_phase = pd.concat(
    [zone_overall, zone_group_stage, zone_knockout_stage],
    ignore_index=True,
)

zone_metrics_path = OUTPUT_DATA_DIR / "final_third_zone_metrics_by_phase.csv"
zone_metrics_by_phase.to_csv(zone_metrics_path, index=False, encoding="utf-8-sig")

print(f"[Check] Saved zone metrics: {zone_metrics_path}")
display(
    zone_metrics_by_phase
    .groupby("competition_phase")
    .agg(
        teams=("team_name", "nunique"),
        avg_matches=("matches", "mean"),
        avg_wide_share=("wide_entry_share", "mean"),
        avg_halfspace_share=("halfspace_entry_share", "mean"),
        avg_central_share=("central_entry_share", "mean"),
    )
    .round(2)
    .reset_index()
)




## Pitch Visualization Design

The pitch is shown from the attacking team's perspective, with final-third channels drawn across the attacking third.

Important note: FIFA's channel labels are already team-oriented. The chart should be read as attacking-channel tendency, not fixed broadcast left/right dominance across all matches.


In [ ]:
"""
Step 3: Helper function for mplsoccer full-pitch final-third overlays
====================================================================
Report layout version:
- Uses a large mplsoccer full pitch as the main visual.
- Keeps titles, color scale, and notes outside the pitch.
- Adds a compact right-side value panel so pitch labels stay readable.
"""

from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Rectangle, FancyBboxPatch
import matplotlib.patheffects as path_effects

channel_order = [
    "left_channel",
    "left_inside_channel",
    "central_channel",
    "right_inside_channel",
    "right_channel",
]

channel_labels = {
    "left_channel": "Left channel",
    "left_inside_channel": "Left half-space",
    "central_channel": "Central",
    "right_inside_channel": "Right half-space",
    "right_channel": "Right channel",
}

# StatsBomb pitch: x 0-120, y 0-80. The final third is x 80-120.
FINAL_THIRD_X0 = 80
FINAL_THIRD_WIDTH = 40
PITCH_HEIGHT = 80
CHANNEL_HEIGHT = PITCH_HEIGHT / 5

REPORT_CMAPS = {
    "entry_share": LinearSegmentedColormap.from_list(
        "entry_share",
        ["#e5f4ec", "#a7d4bf", "#4aa184", "#0f6b57"],
    ),
    "dominance": LinearSegmentedColormap.from_list(
        "dominance",
        ["#f2cf8f", "#d7cf9d", "#7db9a8", "#0f6b57"],
    ),
}


def draw_final_third_zone_map(
    values,
    title,
    subtitle,
    output_path,
    value_suffix="%",
    cmap_name="entry_share",
    vmin=None,
    vmax=None,
    balance_line=None,
):
    values = pd.Series(values).reindex(channel_order).astype(float)

    if vmin is None:
        vmin = float(values.min())
    if vmax is None:
        vmax = float(values.max())
    if np.isclose(vmin, vmax):
        vmax = vmin + 1

    cmap = REPORT_CMAPS[cmap_name] if cmap_name in REPORT_CMAPS else plt.get_cmap(cmap_name)
    norm = plt.Normalize(vmin=vmin, vmax=vmax)

    fig = plt.figure(figsize=(15.5, 7.8), dpi=150, facecolor="#f7f6f2")
    gs = fig.add_gridspec(
        nrows=3,
        ncols=2,
        height_ratios=[0.14, 0.74, 0.12],
        width_ratios=[0.80, 0.20],
        left=0.050,
        right=0.970,
        top=0.965,
        bottom=0.055,
        wspace=0.08,
        hspace=0.03,
    )

    ax_title = fig.add_subplot(gs[0, :])
    ax_pitch = fig.add_subplot(gs[1, 0])
    ax_side = fig.add_subplot(gs[1, 1])
    ax_note = fig.add_subplot(gs[2, :])

    for text_ax in [ax_title, ax_side, ax_note]:
        text_ax.set_axis_off()

    ax_title.text(0.00, 0.82, title, ha="left", va="top", fontsize=18.5, color="#101820", fontweight="medium")
    ax_title.text(0.00, 0.43, subtitle, ha="left", va="top", fontsize=10.5, color="#536173")

    # Dedicated color scale in the title row, away from the pitch.
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cax = ax_title.inset_axes([0.69, 0.55, 0.23, 0.16])
    cbar = fig.colorbar(sm, cax=cax, orientation="horizontal")
    cbar.ax.tick_params(labelsize=8, length=2, colors="#384250")
    cbar.outline.set_visible(False)
    if balance_line is not None:
        cbar.set_label(f"Balance = {balance_line:.0f}%", fontsize=8.2, color="#536173", labelpad=2)

    pitch = Pitch(
        pitch_type="statsbomb",
        half=False,
        pitch_color="#2f6b4f",
        line_color="#f4fff6",
        linewidth=1.55,
        line_zorder=8,
        pad_left=2,
        pad_right=2,
        pad_top=2,
        pad_bottom=7,
    )
    pitch.draw(ax=ax_pitch)
    ax_pitch.set_facecolor("#2f6b4f")

    # Slightly darker non-final-third area to make the final third read as the focus zone.
    ax_pitch.add_patch(
        Rectangle((0, 0), FINAL_THIRD_X0, 80, facecolor="#24583f", edgecolor="none", alpha=0.40, zorder=1)
    )

    ax_pitch.add_patch(
        Rectangle(
            (FINAL_THIRD_X0, 0),
            FINAL_THIRD_WIDTH,
            80,
            facecolor="none",
            edgecolor="#f4fff6",
            linewidth=1.8,
            alpha=0.98,
            zorder=9,
        )
    )

    for i, channel in enumerate(channel_order):
        y0 = i * CHANNEL_HEIGHT
        value = float(values[channel])
        color = cmap(norm(value))

        ax_pitch.add_patch(
            Rectangle(
                (FINAL_THIRD_X0, y0),
                FINAL_THIRD_WIDTH,
                CHANNEL_HEIGHT,
                facecolor=color,
                edgecolor="#f4fff6",
                linewidth=1.2,
                alpha=0.78,
                zorder=4,
            )
        )

        label_effect = [path_effects.withStroke(linewidth=2.4, foreground="#153827", alpha=0.90)]
        ax_pitch.text(
            FINAL_THIRD_X0 + 4.0,
            y0 + CHANNEL_HEIGHT / 2,
            channel_labels[channel],
            ha="left",
            va="center",
            fontsize=9.5,
            color="#ffffff",
            fontweight="medium",
            zorder=10,
            path_effects=label_effect,
        )
        ax_pitch.text(
            FINAL_THIRD_X0 + FINAL_THIRD_WIDTH - 4.0,
            y0 + CHANNEL_HEIGHT / 2,
            f"{value:.1f}{value_suffix}",
            ha="right",
            va="center",
            fontsize=15.0,
            color="#ffffff",
            fontweight="bold",
            zorder=10,
            path_effects=label_effect,
        )

    ax_pitch.annotate(
        "Attacking direction",
        xy=(116, -5.2),
        xytext=(18, -5.2),
        arrowprops=dict(arrowstyle="->", color="#d9f7e6", lw=1.35),
        ha="left",
        va="center",
        fontsize=9.0,
        color="#d9f7e6",
        annotation_clip=False,
        zorder=12,
    )

    # Right-side value panel: keeps the pitch clean while preserving exact labels.
    ax_side.set_xlim(0, 1)
    ax_side.set_ylim(0, 1)
    ax_side.text(0.03, 0.97, "Channel values", ha="left", va="top", fontsize=12.0, color="#101820", fontweight="medium")
    ax_side.text(0.03, 0.925, "Top to bottom follows the pitch", ha="left", va="top", fontsize=8.6, color="#667085")

    ranked = values.sort_values(ascending=False)
    max_value = max(float(values.max()), 1.0)
    bar_x0 = 0.03
    bar_max_width = 0.62
    value_x = 0.96
    y = 0.80

    for channel, value in ranked.items():
        bar_w = bar_max_width * (float(value) / max_value)
        color = cmap(norm(float(value)))

        ax_side.text(
            bar_x0,
            y + 0.052,
            channel_labels[channel],
            ha="left",
            va="bottom",
            fontsize=9.0,
            color="#344054",
        )
        ax_side.text(
            value_x,
            y + 0.052,
            f"{value:.1f}{value_suffix}",
            ha="right",
            va="bottom",
            fontsize=9.0,
            color="#101820",
            fontweight="medium",
        )

        # Keep the bar track short enough that it never collides with the value label.
        ax_side.add_patch(
            FancyBboxPatch(
                (bar_x0, y),
                bar_max_width,
                0.032,
                boxstyle="round,pad=0.003,rounding_size=0.006",
                facecolor="#e6ebe7",
                edgecolor="none",
                alpha=0.85,
            )
        )
        ax_side.add_patch(
            FancyBboxPatch(
                (bar_x0, y),
                bar_w,
                0.032,
                boxstyle="round,pad=0.003,rounding_size=0.006",
                facecolor=color,
                edgecolor="none",
                alpha=0.95,
            )
        )
        y -= 0.145

    ax_side.text(
        0.03,
        0.08,
        "These are entry counts by FIFA channel,\nnot continuous territory control.",
        ha="left",
        va="bottom",
        fontsize=8.3,
        color="#667085",
        linespacing=1.25,
    )

    footnote = (
        "Full-pitch overlay built with mplsoccer. Values are entry-based final-third zone metrics, not continuous possession control.\n"
        "Data source: FIFA Match Centre | Full FIFA Official Stats only | Belgium vs Egypt excluded due to Live Statistics only."
    )
    ax_note.text(0.00, 0.56, footnote, fontsize=8.4, color="#667085", ha="left", va="center")

    fig.savefig(output_path, dpi=240, facecolor=fig.get_facecolor())
    plt.show()

    print(f"Saved figure: {output_path}")





## Visualization 1: Spain Final Third Entry Routes

This pitch map shows Spain's final-third entry distribution across the five FIFA channels.


In [ ]:
"""
Step 4: Spain final-third entry route map
=========================================
"""

plot_phase = "Overall"
# plot_phase = "Group Stage"
# plot_phase = "Knockout Stage"

spain_row = zone_metrics_by_phase[
    (zone_metrics_by_phase["competition_phase"] == plot_phase)
    & (zone_metrics_by_phase["team_name"] == CHAMPION_TEAM)
].iloc[0]

spain_channel_shares = {
    channel: spain_row[f"{channel}_share"]
    for channel in channel_order
}

output_path = OUTPUT_FIG_DIR / f"12_mplsoccer_spain_final_third_entry_routes_{plot_phase.lower().replace(' ', '_')}.png"

draw_final_third_zone_map(
    values=spain_channel_shares,
    title=f"Spain Final Third Entry Routes ({plot_phase})",
    subtitle="Share of Spain's final-third entries by FIFA channel",
    output_path=output_path,
    value_suffix="%",
    cmap_name="entry_share",
    vmin=0,
    vmax=max(35, max(spain_channel_shares.values())),
)

print("Spain channel shares:")
display(pd.DataFrame([spain_channel_shares]).round(2))




## Visualization 2: Tournament Average Entry Routes

This map shows the tournament average final-third entry route distribution. It provides context for judging whether Spain's profile was typical or distinctive.


In [ ]:
"""
Step 5: Tournament average final-third entry route map
======================================================
"""

plot_phase = "Overall"
# plot_phase = "Group Stage"
# plot_phase = "Knockout Stage"

phase_df = zone_metrics_by_phase[zone_metrics_by_phase["competition_phase"] == plot_phase].copy()

# Tournament average by team share: every team receives equal weight.
tournament_channel_shares = {
    channel: phase_df[f"{channel}_share"].mean()
    for channel in channel_order
}

output_path = OUTPUT_FIG_DIR / f"13_mplsoccer_tournament_average_final_third_entry_routes_{plot_phase.lower().replace(' ', '_')}.png"

draw_final_third_zone_map(
    values=tournament_channel_shares,
    title=f"Tournament Average Final Third Entry Routes ({plot_phase})",
    subtitle="Average team share of final-third entries by FIFA channel",
    output_path=output_path,
    value_suffix="%",
    cmap_name="entry_share",
    vmin=0,
    vmax=max(35, max(tournament_channel_shares.values())),
)

print("Tournament average channel shares:")
display(pd.DataFrame([tournament_channel_shares]).round(2))




## Visualization 3: Spain Zone Dominance Proxy

Zone Dominance Proxy compares Spain's zone entries against their opponents' entries in the same grouped zone.

- 50% = balance line
- above 50% = Spain entered that grouped zone more often than opponents
- below 50% = opponents entered that grouped zone more often


In [ ]:
"""
Step 6: Spain grouped Zone Dominance Proxy map
==============================================
"""

plot_phase = "Overall"
spain_row = zone_metrics_by_phase[
    (zone_metrics_by_phase["competition_phase"] == plot_phase)
    & (zone_metrics_by_phase["team_name"] == CHAMPION_TEAM)
].iloc[0]

# Spread grouped dominance values over five visible channels.
# Wide value is shown in both wide channels; half-space value in both half-space channels.
spain_zone_dominance = {
    "left_channel": spain_row["wide_zone_dominance_proxy"],
    "left_inside_channel": spain_row["halfspace_zone_dominance_proxy"],
    "central_channel": spain_row["central_zone_dominance_proxy"],
    "right_inside_channel": spain_row["halfspace_zone_dominance_proxy"],
    "right_channel": spain_row["wide_zone_dominance_proxy"],
}

output_path = OUTPUT_FIG_DIR / f"14_mplsoccer_spain_zone_dominance_proxy_{plot_phase.lower().replace(' ', '_')}.png"

draw_final_third_zone_map(
    values=spain_zone_dominance,
    title=f"Spain Final Third Zone Dominance Proxy ({plot_phase})",
    subtitle="Share of both teams' entries by grouped zone: wide, half-space, central",
    output_path=output_path,
    value_suffix="%",
    cmap_name="dominance",
    vmin=30,
    vmax=75,
    balance_line=50,
)

zone_dominance_summary = pd.DataFrame([
    {
        "team_name": CHAMPION_TEAM,
        "competition_phase": plot_phase,
        "wide_zone_dominance_proxy": spain_row["wide_zone_dominance_proxy"],
        "halfspace_zone_dominance_proxy": spain_row["halfspace_zone_dominance_proxy"],
        "central_zone_dominance_proxy": spain_row["central_zone_dominance_proxy"],
        "field_tilt_proxy": spain_row["field_tilt_proxy"],
    }
])

zone_dominance_path = OUTPUT_DATA_DIR / f"spain_zone_dominance_proxy_{plot_phase.lower().replace(' ', '_')}.csv"
zone_dominance_summary.to_csv(zone_dominance_path, index=False, encoding="utf-8-sig")

display(zone_dominance_summary.round(2))
print(f"Saved data: {zone_dominance_path}")




## Draft Interpretation

Use this section as report-writing support. Edit the text after reviewing the charts.


In [ ]:
"""
Step 7: Generate concise draft takeaways
========================================
"""

report_phase = "Overall"
spain = zone_metrics_by_phase[
    (zone_metrics_by_phase["competition_phase"] == report_phase)
    & (zone_metrics_by_phase["team_name"] == CHAMPION_TEAM)
].iloc[0]
phase_df = zone_metrics_by_phase[zone_metrics_by_phase["competition_phase"] == report_phase]

spain_grouped = {
    "Wide Entry Share": spain["wide_entry_share"],
    "Half-space Entry Share": spain["halfspace_entry_share"],
    "Central Entry Share": spain["central_entry_share"],
}

avg_grouped = {
    "Wide Entry Share": phase_df["wide_entry_share"].mean(),
    "Half-space Entry Share": phase_df["halfspace_entry_share"].mean(),
    "Central Entry Share": phase_df["central_entry_share"].mean(),
}

print("Final Third Zone Dominance Draft Notes")
print("=" * 70)
print(f"Phase: {report_phase}")

print("\nSpain grouped entry shares:")
for key, value in spain_grouped.items():
    print(f"- {key}: {value:.2f}% | tournament avg: {avg_grouped[key]:.2f}%")

print("\nSpain grouped Zone Dominance Proxy:")
print(f"- Wide: {spain['wide_zone_dominance_proxy']:.2f}%")
print(f"- Half-space: {spain['halfspace_zone_dominance_proxy']:.2f}%")
print(f"- Central: {spain['central_zone_dominance_proxy']:.2f}%")

print("\nSuggested interpretation:")
print(
    "The zone-entry view shows where Spain accessed the final third rather than simply how often they got there. "
    "Compared with the tournament average, Spain's route profile can be read through the balance between wide, half-space, and central entries. "
    "Zone Dominance Proxy then adds opponent context by showing whether Spain entered each grouped zone more often than their opponents."
)

print("\nCaution:")
print(
    "These metrics are entry-based. They do not prove sustained possession control in a zone, and they do not directly measure chance quality."
)


